# Chatbot Response Ranking using Embeddings

A case study implementing a response-ranking pipeline for dialogue systems using sentence embeddings and cosine similarity.

**Pipeline:**
1. Load dataset (synthetic or real HuggingFace)
2. Embed contexts and candidate responses with SentenceTransformers
3. Rank candidates by cosine similarity
4. Evaluate with MRR and Recall@k
5. Compare against a random baseline and visualize results

**Evaluation protocol** matches Ubuntu Corpus v2 / DSTC7: one gold response + N distractors drawn from other conversations.

## Table of Contents
1. [Setup & Imports](#1-setup--imports)
2. [Data Structures](#2-data-structures)
3. [Synthetic Dataset — 50 conversations, 10 domains](#3-synthetic-dataset)
4. [Embedder Module](#4-embedder-module)
5. [Ranker Module](#5-ranker-module)
6. [Evaluator Module](#6-evaluator-module)
7. [Real Dataset Loaders (HuggingFace)](#7-real-dataset-loaders)
8. [Full Pipeline](#8-full-pipeline)
9. [Experiments](#9-experiments)

---
## 1. Setup & Imports

In [ ]:
!pip install sentence-transformers numpy pandas tqdm matplotlib scikit-learn datasets -q

In [ ]:
import re
import random
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Union

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
from sentence_transformers import SentenceTransformer

random.seed(42)
np.random.seed(42)
print('All imports successful.')

---
## 2. Data Structures

`ConversationExample` holds a multi-turn context, the gold (correct) response, and a list of distractors. The `candidates()` method returns a shuffled pool with the index of the gold response.

In [ ]:
@dataclass
class ConversationExample:
    context: str
    gold: str
    distractors: List[str] = field(default_factory=list)

    def candidates(self, shuffle: bool = True) -> Tuple[List[str], int]:
        """Return (candidate_list, gold_index). Gold is always included."""
        pool = [self.gold] + self.distractors
        if shuffle:
            random.shuffle(pool)
        gold_idx = pool.index(self.gold)
        return pool, gold_idx


def get_pool_size(examples: List[ConversationExample]) -> int:
    return 1 + len(examples[0].distractors)

---
## 3. Synthetic Dataset

50 hand-crafted multi-turn conversations across 10 domains:
**IT Support · Customer Service · Health/Medical · Education · Cooking · Finance · Travel · Fitness · Chitchat · Mental Health**

Disstractors are gold responses from *other* conversations — the standard negative-sampling strategy used in Ubuntu Corpus and DSTC7.

In [ ]:
_RAW: List[Tuple[List[str], str]] = [

    # ── IT Support ──────────────────────────────────────────────────────────
    (
        ["User: My laptop screen goes black randomly and won't wake up.",
         "Agent: Does it happen only on battery or also when plugged in?",
         "User: Both. It started after a Windows update last week."],
        "Agent: A recent Windows update could have changed your power or display driver settings. "
        "Go to Device Manager, expand Display Adapters, right-click your GPU, and select "
        "'Update driver'. Also check Power Options and disable 'Adaptive brightness'.",
    ),
    (
        ["User: My WiFi keeps disconnecting every 10 minutes.",
         "Agent: Is it only on your device or do others disconnect too?",
         "User: Only on my laptop. Phone and tablet stay connected fine."],
        "Agent: Since only your laptop disconnects, try disabling WiFi power management. "
        "In Device Manager, right-click your network adapter, go to Properties \u2192 "
        "Power Management, and uncheck 'Allow the computer to turn off this device to save power'.",
    ),
    (
        ["User: I accidentally deleted an important folder. Can I recover it?",
         "Agent: Did you empty the Recycle Bin after deleting it?",
         "User: Yes, I emptied it already."],
        "Agent: Don't write any new files to that drive to avoid overwriting the data. "
        "Use a recovery tool like Recuva or TestDisk. Both are free and can often "
        "retrieve files even after the Recycle Bin is emptied.",
    ),
    (
        ["User: My SSD is not detected in BIOS after I reseated it.",
         "Agent: Which slot did you put it back into?",
         "User: The same M.2 slot it was in before."],
        "Agent: Try clearing the CMOS by removing the motherboard battery for 30 seconds. "
        "Also check that the M.2 screw is firmly tightening the drive at a slight angle \u2014 "
        "a loose screw is a common cause of intermittent detection.",
    ),
    (
        ["User: Python crashes with 'MemoryError' when processing a large CSV.",
         "Agent: How large is the CSV and how much RAM does your machine have?",
         "User: It's 8 GB and I have 16 GB RAM."],
        "Agent: Loading an 8 GB CSV entirely into pandas can exceed available RAM due to overhead. "
        "Use `pd.read_csv(..., chunksize=100000)` to process the file in chunks, "
        "or switch to Polars or Dask which handle large files more efficiently.",
    ),

    # ── Customer Service ─────────────────────────────────────────────────────
    (
        ["Customer: I ordered a jacket 10 days ago and tracking still shows 'processing'.",
         "Agent: I'm sorry for the delay. May I have your order number?",
         "Customer: It's ORD-884521."],
        "Agent: Thank you. I can see your order was held due to a stock discrepancy. "
        "I've escalated it to our warehouse team and you'll receive a dispatch "
        "confirmation within 24 hours. We'll also apply a 10% discount to your next order.",
    ),
    (
        ["Customer: I was charged twice for the same subscription.",
         "Agent: I apologize for that. Can you confirm the email on the account?",
         "Customer: Yes, it's john.doe@email.com."],
        "Agent: I can confirm a duplicate charge on 14 April. I've initiated a full refund "
        "for the duplicate transaction \u2014 it should appear on your statement within 3\u20135 business days. "
        "You'll receive a confirmation email shortly.",
    ),
    (
        ["Customer: The product I received is completely different from what I ordered.",
         "Agent: I'm very sorry. Could you describe what you received?",
         "Customer: I ordered a blue mug but got a red vase."],
        "Agent: That is clearly a picking error on our end. I'll send a prepaid return label "
        "to your email immediately, and ship the correct blue mug via express delivery at no cost. "
        "You don't need to wait for us to receive the return before we dispatch the replacement.",
    ),
    (
        ["Customer: My promo code says it's invalid at checkout.",
         "Agent: Is the code case-sensitive when you enter it?",
         "Customer: I've tried both uppercase and lowercase."],
        "Agent: The code SUMMER20 expired on April 15th. As a courtesy I'll manually apply "
        "a 20% discount to your current cart. Please don't close the session and "
        "I'll update the order on my end right now.",
    ),
    (
        ["Customer: I want to cancel my gym membership but the app won't let me.",
         "Agent: Are you seeing a specific error message or just no cancel button?",
         "Customer: There's just no cancel button anywhere in settings."],
        "Agent: Cancellation must currently be done through our front desk or by calling "
        "our support line at 1-800-555-0199 during business hours. I know that's not ideal \u2014 "
        "we're adding in-app cancellation next month. I can also email you a cancellation form right now.",
    ),

    # ── Health & Medical ──────────────────────────────────────────────────────
    (
        ["Patient: I've had a headache every morning for the past week.",
         "Doctor: Is the pain worse on one side or both?",
         "Patient: Mostly on the right side, behind my eye."],
        "Doctor: A unilateral headache behind the eye, especially recurring in the morning, "
        "can indicate cluster headaches or tension-type headaches. I'd like to check your "
        "blood pressure today and ask about your sleep quality. Have you changed your screen time recently?",
    ),
    (
        ["Patient: I've been feeling very tired even after 8 hours of sleep.",
         "Doctor: How long has this been going on?",
         "Patient: About three months now. Also gained some weight."],
        "Doctor: Fatigue combined with unexplained weight gain over three months warrants "
        "a thyroid panel. Hypothyroidism fits both symptoms very well. "
        "I'll order a TSH, Free T4, and a CBC to rule out anaemia as well.",
    ),
    (
        ["Patient: My lower back has been aching since I started working from home.",
         "Doctor: Do you have any numbness or tingling down your legs?",
         "Patient: No, just the dull ache across my lower back."],
        "Doctor: This sounds like postural lower back pain \u2014 very common with prolonged desk work. "
        "I recommend raising your monitor to eye level, using a lumbar-support cushion, "
        "and doing 5 minutes of hip flexor stretches every hour. I can also refer you to physiotherapy.",
    ),
    (
        ["Patient: I have a rash on my arm that appeared two days ago. It's itchy.",
         "Doctor: Is the rash spreading or staying in the same spot?",
         "Patient: It's spreading slowly toward my elbow."],
        "Doctor: A spreading itchy rash could be contact dermatitis, ringworm, or in some cases "
        "an early sign of Lyme disease if you've been outdoors recently. "
        "I'd like to examine it in person \u2014 can you come in today or tomorrow? "
        "In the meantime, avoid scratching and apply a cool compress.",
    ),
    (
        ["Patient: Should I take antibiotics for a cold I've had for five days?",
         "Doctor: Are you experiencing green/yellow mucus, sore throat, or fever?",
         "Patient: Just a runny nose and congestion, no fever."],
        "Doctor: Colds are caused by viruses, so antibiotics won't help and can cause side effects. "
        "Stay hydrated, use saline nasal rinse, and rest. "
        "If symptoms worsen or a fever develops after day 7, come back in \u2014 "
        "that may signal a secondary bacterial infection.",
    ),

    # ── Education / Tutoring ─────────────────────────────────────────────────
    (
        ["Student: I don't understand when to use the definite article 'the' in English.",
         "Tutor: Can you give me a sentence where you're unsure?",
         "Student: 'I saw the dog' vs 'I saw a dog' \u2014 when do I use which?"],
        "Tutor: Great example. Use 'a dog' the first time you mention it \u2014 it's new information. "
        "Use 'the dog' afterwards because both speaker and listener now know which dog. "
        "Think of 'the' as pointing to something already established in shared context.",
    ),
    (
        ["Student: I keep mixing up correlation and causation in statistics.",
         "Tutor: That's a very common confusion. What's your current understanding?",
         "Student: I think correlation means two things happen together?"],
        "Tutor: Exactly \u2014 correlation means two variables move together, but one doesn't cause the other. "
        "Classic example: ice cream sales and drowning rates both rise in summer. "
        "The hidden cause is hot weather. Always ask: 'is there a third variable explaining both?' "
        "before claiming causation.",
    ),
    (
        ["Student: I don't know how to start my essay introduction.",
         "Tutor: What is your essay topic?",
         "Student: The impact of social media on teenagers' mental health."],
        "Tutor: Open with a hook \u2014 a striking statistic works well here, like 'teens who spend "
        "5+ hours daily on social media are 3\u00d7 more likely to report depression'. "
        "Then narrow down to your thesis. Avoid starting with 'In today's world...' \u2014 "
        "it's overused and weakens your opening.",
    ),
    (
        ["Student: I'm really struggling with integration by parts.",
         "Tutor: Do you remember the formula: \u222fu dv = uv - \u222fv du?",
         "Student: I know the formula but I don't know how to pick u and v."],
        "Tutor: Use the LIATE rule to pick u: Logarithms, Inverse trig, Algebraic (polynomials), "
        "Trig, Exponential \u2014 choose u from whichever comes first in that list. "
        "For \u222fx\u00b7e\u02e3 dx, x is algebraic and e\u02e3 is exponential, so u = x, dv = e\u02e3 dx.",
    ),
    (
        ["Student: How do I stop procrastinating on revision?",
         "Tutor: Have you tried breaking study sessions into smaller blocks?",
         "Student: I start strong but then get distracted after 10 minutes."],
        "Tutor: 10 minutes is a sign your tasks feel too large and overwhelming. "
        "Try the Pomodoro technique: 25-minute focused sprint, 5-minute break. "
        "More importantly, write down the single next action before you start \u2014 "
        "'read page 45 to 60' beats 'study Chapter 3'.",
    ),

    # ── Cooking ──────────────────────────────────────────────────────────────
    (
        ["User: My pasta sauce always tastes too acidic.",
         "Chef: Are you using canned tomatoes or fresh ones?",
         "User: Canned. San Marzano brand."],
        "Chef: San Marzano are great but still acidic. Add a small pinch of baking soda "
        "(\u00bc tsp per 400 g can) right after the tomatoes go in \u2014 it neutralizes acidity without "
        "adding sweetness. Alternatively, simmer for 45+ minutes; long cooking mellows acidity naturally.",
    ),
    (
        ["User: My scrambled eggs turn out rubbery.",
         "Chef: What heat are you cooking them on?",
         "User: Medium-high, and I add milk."],
        "Chef: Medium-high is too hot \u2014 it makes proteins seize up fast. Switch to low heat, "
        "use a rubber spatula, and stir constantly. Pull them off the heat when they still look "
        "slightly underdone; carry-over heat finishes them. Skip the milk \u2014 it dilutes flavour and adds steam.",
    ),
    (
        ["User: How do I get a crispy crust on pan-seared chicken?",
         "Chef: Is your pan preheated before you add the chicken?",
         "User: I usually put the chicken in while the pan is heating."],
        "Chef: That's the issue. Preheat a stainless or cast-iron pan over medium-high for 2 minutes "
        "until a drop of water skips across. Pat the chicken completely dry, season, then lay it "
        "skin-side down without touching it for 5\u20136 minutes. Resist moving it \u2014 it releases naturally when crust forms.",
    ),
    (
        ["User: My bread doesn't rise properly. It stays flat.",
         "Chef: What yeast are you using and how old is it?",
         "User: Instant yeast, opened a few months ago."],
        "Chef: Old yeast is the likely culprit. Test it: dissolve 1 tsp in warm water (40\u00b0C) "
        "with a pinch of sugar \u2014 it should foam in 10 minutes. If it doesn't, the yeast is dead. "
        "Also ensure your water isn't too hot (above 50\u00b0C kills yeast).",
    ),
    (
        ["User: Can I substitute Greek yogurt for sour cream in baking?",
         "Chef: Are you using it in a dip or baking something?",
         "User: In a chocolate cake recipe."],
        "Chef: Yes, full-fat Greek yogurt is a 1:1 substitute in baking. "
        "The fat content keeps the cake moist and the acidity reacts with baking soda "
        "just like sour cream. Low-fat yogurt works too but may produce a slightly denser crumb.",
    ),

    # ── Finance ──────────────────────────────────────────────────────────────
    (
        ["Client: I'm 28 and just started investing. Where should I begin?",
         "Advisor: Do you have an emergency fund of 3\u20136 months' expenses first?",
         "Client: Yes, I have 4 months saved up."],
        "Advisor: Great foundation. At 28 with a long horizon, prioritize your employer's 401(k) "
        "up to the full match \u2014 that's a guaranteed 50\u2013100% return. Then max your Roth IRA ($7,000/yr). "
        "For both, use low-cost total-market index funds; Vanguard VTSAX or Fidelity FZROX are solid choices.",
    ),
    (
        ["Client: My credit score dropped 40 points after paying off a loan.",
         "Advisor: Did you close the account after paying it off?",
         "Client: Yes, I called and closed it right away."],
        "Advisor: Closing the account reduced your 'length of credit history' and "
        "available credit, both of which affect your score. For installment loans, "
        "paying them off is still correct \u2014 the score dip is temporary (3\u20136 months) "
        "and your overall credit profile will improve.",
    ),
    (
        ["Client: Should I pay off my student loans early or invest the extra money?",
         "Advisor: What's your interest rate on the loans?",
         "Client: 5.5% fixed."],
        "Advisor: At 5.5%, the decision is close. If your investments can realistically earn 7\u20138% "
        "(historical S&P 500 average), investing wins mathematically in the long run. "
        "A common middle ground: invest up to any employer 401(k) match, then split extra cash "
        "50/50 between the loan and a Roth IRA.",
    ),
    (
        ["Client: I keep overspending and running out of money before month end.",
         "Advisor: Do you track your spending in any way currently?",
         "Client: Not really, I just watch my bank balance."],
        "Advisor: Watching the balance isn't enough because it doesn't show categories. "
        "Try the 50/30/20 rule: 50% needs (rent, food), 30% wants (dining, entertainment), "
        "20% savings/debt. Tools like YNAB or Monarch Money auto-categorize transactions \u2014 "
        "most people are shocked where their money actually goes.",
    ),
    (
        ["Client: Is it safe to keep all my savings in one bank?",
         "Advisor: How much total savings are you holding?",
         "Client: Around $180,000."],
        "Advisor: FDIC insurance covers up to $250,000 per depositor per bank, so $180,000 "
        "in one FDIC-insured bank is fully protected. However, spreading across two banks "
        "gives you redundancy if one has a service outage or account freeze issue \u2014 "
        "a practical precaution regardless of insurance limits.",
    ),

    # ── Travel ───────────────────────────────────────────────────────────────
    (
        ["Traveler: I'm visiting Japan for the first time in April. Any tips?",
         "Guide: How long will you be there and which cities?",
         "Traveler: Two weeks \u2014 Tokyo, Kyoto, and Osaka."],
        "Guide: Perfect itinerary. April is cherry blossom season \u2014 book accommodation now, "
        "prices triple and rooms sell out. Get an IC Card (Suica or ICOCA) at the airport "
        "for seamless transit. In Kyoto, rent a bicycle to reach Fushimi Inari at dawn "
        "before crowds arrive. Reserve a kaiseki dinner in advance; they fill up fast.",
    ),
    (
        ["Traveler: What's the safest way to carry money abroad?",
         "Guide: Are you going to a region with good ATM coverage?",
         "Traveler: Visiting rural Southeast Asia, so ATMs may be sparse."],
        "Guide: Carry a mix: use a Charles Schwab debit card (reimburses ATM fees worldwide) "
        "for towns with ATMs, and keep 3\u20134 days of cash in local currency as backup. "
        "Split cash between a money belt and your bag \u2014 never keep it all together. "
        "Avoid airport exchange kiosks; they charge 10\u201315% above market rate.",
    ),
    (
        ["Traveler: I got food poisoning abroad. What should I do?",
         "Guide: Are you able to keep fluids down?",
         "Traveler: Barely. Everything I drink comes back up."],
        "Guide: If you can't keep fluids down for 12+ hours, see a doctor \u2014 dehydration risk "
        "is serious in hot climates. Most hotels have a doctor on call. "
        "Oral rehydration salts (ORS) work faster than plain water; pharmacies carry them. "
        "Avoid anti-diarrheal medication in the first 24 hours unless a doctor prescribes it.",
    ),
    (
        ["Traveler: My flight got cancelled and the airline is offering a full refund or a voucher.",
         "Guide: Is the voucher worth significantly more than the refund?",
         "Traveler: Voucher is $50 more but expires in 6 months."],
        "Guide: Take the refund unless you're certain you'll fly that airline within 6 months. "
        "Voucher blackout dates and restrictions often make the extra $50 worthless. "
        "Under EC 261 (if flying from or to the EU), you may also be entitled to additional "
        "compensation depending on notice time and distance.",
    ),
    (
        ["Traveler: How do I avoid jet lag flying from New York to Tokyo?",
         "Guide: What time does your flight depart?",
         "Traveler: Evening flight, landing Tokyo morning local time."],
        "Guide: On the plane, set your watch to Tokyo time immediately and sleep "
        "during Tokyo's night hours \u2014 use an eye mask and earplugs. "
        "After landing, stay awake until 9\u201310 PM local time regardless of tiredness; "
        "get sunlight exposure in the afternoon, which resets your circadian clock fastest.",
    ),

    # ── Fitness ──────────────────────────────────────────────────────────────
    (
        ["User: I've been running 5k three times a week but my time isn't improving.",
         "Coach: How long have you been at this routine?",
         "User: About four months."],
        "Coach: Four months without progression means your body has adapted \u2014 you need new stimulus. "
        "Add one interval session per week: 8\u00d7400 m at hard effort with 90-second recoveries. "
        "Keep your other two runs easy (conversational pace). Speed work is what actually improves 5k time.",
    ),
    (
        ["User: I want to build muscle but I don't want to get too bulky.",
         "Coach: Are you currently doing any resistance training?",
         "User: No, just cardio. I'm afraid of lifting weights."],
        "Coach: 'Getting bulky' requires years of consistent heavy lifting plus a significant calorie surplus \u2014 "
        "it won't happen by accident. Resistance training with moderate weight and 8\u201312 reps "
        "builds lean, defined muscle. Start with compound movements: squats, rows, push-ups. "
        "It will make your cardio more effective too.",
    ),
    (
        ["User: My knees hurt when I do squats.",
         "Coach: Does the pain start right away or after a few reps?",
         "User: It starts immediately when I go below parallel."],
        "Coach: Pain below parallel usually means one of three things: tight hip flexors limiting depth, "
        "heels rising (put 10 lb plates under them temporarily to diagnose), "
        "or knees caving inward. Film yourself from the side and front. "
        "In the meantime, try box squats to a depth just above parallel while you address mobility.",
    ),
    (
        ["User: I'm losing weight but I also seem to be losing muscle.",
         "Coach: How large is your daily calorie deficit?",
         "User: I'm eating about 1,000 calories below maintenance."],
        "Coach: A 1,000-calorie deficit is too aggressive and is causing muscle breakdown. "
        "Cut it to 300\u2013500 calories below maintenance. More importantly, increase protein to "
        "at least 2.2 g per kg of bodyweight \u2014 that's the key lever for preserving muscle during a cut. "
        "Keep lifting heavy; don't switch to high-rep 'toning' work.",
    ),
    (
        ["User: I get really sore for 3 days after leg day and can't train.",
         "Coach: Are you new to training or returning after a break?",
         "User: I've been lifting consistently for a year."],
        "Coach: After a year, 3-day soreness suggests you're doing too much volume in one session. "
        "Split leg work across two days (quads/glutes one day, hamstrings/calves another). "
        "Also try 10 minutes of light cycling immediately post-workout \u2014 blood flow "
        "clears metabolic waste and cuts DOMS severity significantly.",
    ),

    # ── Chitchat ─────────────────────────────────────────────────────────────
    (
        ["Alice: I've been reading a lot lately. Any book recommendations?",
         "Bob: What genres do you usually enjoy?",
         "Alice: I love thrillers and anything that makes me think."],
        "Bob: Then you'd love 'Dark Matter' by Blake Crouch \u2014 it's a fast-paced sci-fi thriller "
        "with genuine philosophical depth about identity and choice. "
        "Also try 'The Thursday Murder Club' by Richard Osman if you want something "
        "lighter but still sharp.",
    ),
    (
        ["Alice: I've been really stressed out with work lately.",
         "Bob: That sounds tough. Is it the workload or something else?",
         "Alice: It's mostly my manager \u2014 he micromanages everything."],
        "Bob: Micromanagement is exhausting because it signals a lack of trust. "
        "It sometimes helps to proactively over-communicate your progress before he asks \u2014 "
        "it can reduce his anxiety and give you more autonomy. "
        "Have you tried a brief weekly update email to him outlining what you're working on?",
    ),
    (
        ["Alice: I'm thinking of learning a new language. Which one should I pick?",
         "Bob: Do you have a specific reason \u2014 travel, career, or just personal interest?",
         "Alice: Mainly travel. I love going to Europe."],
        "Bob: Spanish is the most practical for Europe and worldwide \u2014 widely spoken in Spain "
        "and gives you access to Latin America too. "
        "Italian is easier for English speakers and incredibly rewarding in Italy. "
        "If France is your top destination, French pairs well with travel to North Africa as a bonus.",
    ),
    (
        ["Alice: I've been watching too much TV and feeling guilty about it.",
         "Bob: How many hours a day roughly?",
         "Alice: Probably 4\u20135 hours in the evenings."],
        "Bob: That's a significant chunk. Try the 'replacement' trick instead of cutting cold turkey \u2014 "
        "swap one hour of passive watching for something with mild engagement: an audiobook during a walk, "
        "or a podcast while doing chores. You end up with the same downtime but feel less zombie-like.",
    ),
    (
        ["Alice: I just moved to a new city and don't know anyone.",
         "Bob: How long have you been there?",
         "Alice: About two months. I work remotely so I don't meet people at work."],
        "Bob: Remote work makes this genuinely harder \u2014 you have to be intentional. "
        "Meetup.com groups around hobbies (climbing, board games, language exchange) are consistently "
        "the fastest way to meet people with shared interests. "
        "Also look for local sports leagues; recreational volleyball or ultimate frisbee "
        "teams are very welcoming to newcomers.",
    ),

    # ── Mental Health ─────────────────────────────────────────────────────────
    (
        ["Client: I've been feeling really anxious about everything lately.",
         "Therapist: When you say 'everything', can you give me an example from this week?",
         "Client: Even small things like sending an email at work make me spiral."],
        "Therapist: What you're describing \u2014 catastrophizing ordinary tasks \u2014 is very common in "
        "generalized anxiety. One technique that helps is asking yourself: 'What's the worst realistic outcome, "
        "and could I handle it?' Usually the answer to the second question is yes, which interrupts the spiral. "
        "Would you like to practice this with the email example?",
    ),
    (
        ["Client: I feel like I'm always tired and nothing is fun anymore.",
         "Therapist: How long have you been feeling this way?",
         "Client: A few months. I used to love hiking but now I can't be bothered."],
        "Therapist: Losing interest in activities you used to enjoy \u2014 called anhedonia \u2014 "
        "combined with persistent fatigue is something I'd like to explore further. "
        "These can be signs of depression. How is your sleep, and have you noticed any changes in appetite? "
        "I want to make sure we get the full picture before we talk about next steps.",
    ),
    (
        ["Client: I have a big presentation at work and I can't sleep because of it.",
         "Therapist: Is this a recurring pattern before important events?",
         "Client: Yes, I've always been like this before big things."],
        "Therapist: Pre-event insomnia from performance anxiety is very treatable. "
        "Try a 'worry dump' 2 hours before bed: write every concern down and then physically close the notebook. "
        "This externalizes the thoughts so your brain stops rehearsing them. "
        "Also, a 4-7-8 breathing exercise (inhale 4, hold 7, exhale 8) activates your parasympathetic system.",
    ),
    (
        ["Client: I feel guilty spending any time on myself.",
         "Therapist: Where do you think that guilt comes from?",
         "Client: My parents always said that resting was lazy."],
        "Therapist: That's a deeply ingrained belief \u2014 'rest equals laziness' \u2014 and it sounds like "
        "it's been with you since childhood. It's worth examining: do you apply that same standard to people you love? "
        "Self-care isn't a reward for productivity; it's maintenance that makes everything else sustainable. "
        "What would feel like a safe, small act of self-care you could try this week?",
    ),
    (
        ["Client: I snap at my partner over small things and then feel terrible.",
         "Therapist: What's usually happening in the moments just before you snap?",
         "Client: I think I'm already stressed from work and they just add one more thing."],
        "Therapist: That's a classic stress-displacement pattern \u2014 unprocessed work stress "
        "spills over at home because home feels safer to express emotions. "
        "Creating a 10-minute transition ritual when you finish work (walk, tea, music) "
        "can act as a pressure valve before you re-engage with your partner. "
        "Would it help to tell your partner what's going on so they understand the context?",
    ),
]

print(f'Loaded {len(_RAW)} raw conversations across 10 domains.')

In [ ]:
def build_dataset(num_distractors: int = 9) -> List[ConversationExample]:
    """
    Build ConversationExample objects.
    Distractors are gold responses from other conversations —
    a standard negative-sampling strategy used in benchmarks like Ubuntu Corpus.
    """
    examples = []
    all_gold = [gold for _, gold in _RAW]

    for i, (turns, gold) in enumerate(_RAW):
        context = ' | '.join(turns)
        distractor_pool = [g for j, g in enumerate(all_gold) if j != i]
        num_d = min(num_distractors, len(distractor_pool))
        distractors = random.sample(distractor_pool, num_d)
        examples.append(ConversationExample(context=context, gold=gold, distractors=distractors))

    return examples


# Quick sanity check
examples_demo = build_dataset(num_distractors=9)
ex = examples_demo[0]
print(f'Total examples: {len(examples_demo)}')
print(f'Pool size per example: {get_pool_size(examples_demo)}')
print(f'\nSample context (truncated):')
print(f'  {ex.context[:200]}...')
print(f'\nGold response (truncated):')
print(f'  {ex.gold[:150]}...')

---
## 4. Embedder Module

`Embedder` wraps SentenceTransformers and returns **L2-normalised** vectors.
Normalisation makes cosine similarity equivalent to a dot product: `sim(q,d) = q·d`, which enables fast retrieval without the sqrt overhead.

In [ ]:
class Embedder:
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2', device: str = 'cpu'):
        print(f"Loading model '{model_name}' on {device}...")
        self.model = SentenceTransformer(model_name, device=device)
        self.dim = self.model.get_sentence_embedding_dimension()
        print(f'Model ready — embedding dimension: {self.dim}')

    def embed(
        self,
        texts: Union[str, List[str]],
        batch_size: int = 64,
        show_progress: bool = False,
    ) -> np.ndarray:
        """Return L2-normalised embeddings of shape (N, dim)."""
        if isinstance(texts, str):
            texts = [texts]
        embeddings = self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=show_progress,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )
        return embeddings.astype(np.float32)

    def embed_contexts(self, contexts: List[str]) -> np.ndarray:
        return self.embed(contexts, show_progress=True)

    def embed_candidates_batched(
        self,
        candidates_per_example: List[List[str]],
    ) -> List[np.ndarray]:
        """Flatten all candidates, embed in one batch, then split back."""
        all_candidates = [c for pool in candidates_per_example for c in pool]
        flat_embeddings = self.embed(all_candidates, show_progress=True)
        result = []
        offset = 0
        for pool in candidates_per_example:
            n = len(pool)
            result.append(flat_embeddings[offset: offset + n])
            offset += n
        return result

---
## 5. Ranker Module

Ranks candidates by **cosine similarity** (= dot product for L2-normalised vectors).

- `rank_candidates()` — ranks one example's pool
- `rank_all()` — applies ranking to all examples
- `baseline_random_rank()` — computes the analytical expected MRR for a random ranker

In [ ]:
@dataclass
class RankingResult:
    gold_rank: int
    gold_score: float
    ranked_candidates: List[str] = field(default_factory=list)
    ranked_scores: List[float] = field(default_factory=list)

    @property
    def reciprocal_rank(self) -> float:
        return 1.0 / self.gold_rank


def rank_candidates(
    context_emb: np.ndarray,
    candidate_embs: np.ndarray,
    candidates: List[str],
    gold_idx: int,
) -> RankingResult:
    scores = candidate_embs @ context_emb
    ranked_indices = np.argsort(-scores)
    ranked_candidates = [candidates[i] for i in ranked_indices]
    ranked_scores = [float(scores[i]) for i in ranked_indices]
    gold_rank = int(np.where(ranked_indices == gold_idx)[0][0]) + 1
    return RankingResult(
        gold_rank=gold_rank,
        gold_score=float(scores[gold_idx]),
        ranked_candidates=ranked_candidates,
        ranked_scores=ranked_scores,
    )


def rank_all(
    context_embs: np.ndarray,
    candidate_embs_list: List[np.ndarray],
    candidates_list: List[List[str]],
    gold_indices: List[int],
) -> List[RankingResult]:
    results = []
    for ctx_emb, cand_embs, candidates, gold_idx in zip(
        context_embs, candidate_embs_list, candidates_list, gold_indices
    ):
        results.append(rank_candidates(ctx_emb, cand_embs, candidates, gold_idx))
    return results


def baseline_random_rank(num_candidates: int) -> float:
    """Analytical expected MRR for a random ranker: (1/N) * sum(1/k for k in 1..N)."""
    return sum(1.0 / k for k in range(1, num_candidates + 1)) / num_candidates

---
## 6. Evaluator Module

| Metric | Description |
|--------|-------------|
| **MRR** | Mean Reciprocal Rank — average of 1/rank. Range [0,1]; perfect = 1.0 |
| **R@k** | Recall@k — fraction of examples where gold appears in top-k. R@1 = exact accuracy |

In [ ]:
def mean_reciprocal_rank(results: List[RankingResult]) -> float:
    return float(np.mean([r.reciprocal_rank for r in results]))


def recall_at_k(results: List[RankingResult], k: int) -> float:
    return sum(1 for r in results if r.gold_rank <= k) / len(results)


def compute_all_metrics(
    results: List[RankingResult],
    k_values: List[int] = (1, 2, 3, 5),
) -> Dict[str, float]:
    metrics = {'MRR': mean_reciprocal_rank(results)}
    for k in k_values:
        metrics[f'R@{k}'] = recall_at_k(results, k)
    return metrics


def rank_distribution(results: List[RankingResult]) -> Dict[int, int]:
    dist: Dict[int, int] = {}
    for r in results:
        dist[r.gold_rank] = dist.get(r.gold_rank, 0) + 1
    return dict(sorted(dist.items()))


def print_metrics_table(metrics: Dict[str, float], label: str = '') -> None:
    sep = '-' * 20
    if label:
        print(f"\n{'='*20}  {label}  {'='*20}")
    print(f"{'Metric':<10} {'Score':>8}")
    print(sep)
    for name, value in metrics.items():
        print(f'{name:<10} {value:>8.4f}')
    print(sep)


def print_rank_distribution(dist: Dict[int, int], total: int) -> None:
    print(f"\n{'Rank':<6} {'Count':>6}  {'%':>6}  Bar")
    print('-' * 40)
    max_count = max(dist.values()) if dist else 1
    for rank, count in dist.items():
        pct = 100 * count / total
        bar = '#' * int(30 * count / max_count)
        print(f'{rank:<6} {count:>6}  {pct:>5.1f}%  {bar}')

---
## 7. Real Dataset Loaders

Three HuggingFace datasets are supported. All use the same negative-sampling strategy: distractors are gold responses from other conversations in the same split.

| Key | Dataset | Size |
|-----|---------|------|
| `soda` | allenai/soda — social dialogues (Kim et al., 2022) | 1.5M |
| `blended_skill_talk` | Facebook BST — persona + knowledge + empathy (Smith et al., 2020) | 7k |
| `hh_rlhf` | Anthropic HH-RLHF — RLHF preference data (Bai et al., 2022) | 169k |

> **Note:** Run these only if you want to evaluate on real data. They require an internet connection.

In [ ]:
def _sample_distractors(all_gold: List[str], own_index: int, n: int) -> List[str]:
    pool = [g for i, g in enumerate(all_gold) if i != own_index]
    return random.sample(pool, min(n, len(pool)))


def load_soda(
    split: str = 'test',
    max_examples: int = 300,
    num_distractors: int = 9,
    min_turns: int = 3,
    seed: int = 42,
) -> List[ConversationExample]:
    """Load SODA from HuggingFace (allenai/soda)."""
    from datasets import load_dataset
    random.seed(seed)
    print(f"  Downloading 'allenai/soda' ({split} split)...")
    ds = load_dataset('allenai/soda', split=split)
    convs = [ex['dialogue'] for ex in ds if len(ex['dialogue']) >= min_turns]
    random.shuffle(convs)
    if max_examples:
        convs = convs[:max_examples]
    all_gold = [turns[-1].strip() for turns in convs]
    examples = []
    for i, turns in enumerate(convs):
        context = ' | '.join(t.strip() for t in turns[:-1])
        gold = turns[-1].strip()
        distractors = _sample_distractors(all_gold, i, num_distractors)
        examples.append(ConversationExample(context=context, gold=gold, distractors=distractors))
    return examples


def load_blended_skill_talk(
    split: str = 'test',
    max_examples: int = 300,
    num_distractors: int = 9,
    seed: int = 42,
) -> List[ConversationExample]:
    """Load BlendedSkillTalk from HuggingFace."""
    from datasets import load_dataset
    random.seed(seed)
    print(f"  Downloading 'blended_skill_talk' ({split} split)...")
    ds = load_dataset('blended_skill_talk', split=split)
    raw: List[Tuple[List[str], str]] = []
    for ex in ds:
        prev = ex.get('previous_utterance', [])
        free = ex.get('free_messages', [])
        if prev and free:
            raw.append((prev, free[0]))
    random.shuffle(raw)
    if max_examples:
        raw = raw[:max_examples]
    all_gold = [gold.strip() for _, gold in raw]
    examples = []
    for i, (ctx_turns, gold) in enumerate(raw):
        context = ' | '.join(t.strip() for t in ctx_turns)
        gold = gold.strip()
        distractors = _sample_distractors(all_gold, i, num_distractors)
        examples.append(ConversationExample(context=context, gold=gold, distractors=distractors))
    return examples


def _parse_hh_turns(text: str) -> List[str]:
    parts = re.split(r'\n\nHuman:\s*|\n\nAssistant:\s*', text.strip())
    return [p.strip() for p in parts if p.strip()]


def load_hh_rlhf(
    split: str = 'test',
    max_examples: int = 300,
    num_distractors: int = 9,
    min_turns: int = 3,
    seed: int = 42,
) -> List[ConversationExample]:
    """Load Anthropic HH-RLHF from HuggingFace."""
    from datasets import load_dataset
    random.seed(seed)
    print(f"  Downloading 'Anthropic/hh-rlhf' ({split} split)...")
    ds = load_dataset('Anthropic/hh-rlhf', split=split)
    raw: List[Tuple[List[str], str]] = []
    for ex in ds:
        turns = _parse_hh_turns(ex['chosen'])
        if len(turns) < min_turns:
            continue
        raw.append((turns[:-1], turns[-1]))
    random.shuffle(raw)
    if max_examples:
        raw = raw[:max_examples]
    all_gold = [gold for _, gold in raw]
    examples = []
    for i, (ctx_turns, gold) in enumerate(raw):
        context = ' | '.join(ctx_turns)
        distractors = _sample_distractors(all_gold, i, num_distractors)
        examples.append(ConversationExample(context=context, gold=gold, distractors=distractors))
    return examples


DATASET_REGISTRY = {
    'soda': load_soda,
    'blended_skill_talk': load_blended_skill_talk,
    'hh_rlhf': load_hh_rlhf,
}

DATASET_INFO = {
    'soda': 'allenai/soda — 1.5M social dialogues grounded in commonsense knowledge',
    'blended_skill_talk': 'facebook/bst — 7k conversations blending persona, knowledge & empathy',
    'hh_rlhf': 'Anthropic/hh-rlhf — human-assistant dialogues with RLHF preference labels',
}


def load_real_dataset(
    name: str,
    split: str = 'test',
    max_examples: int = 300,
    num_distractors: int = 9,
    seed: int = 42,
) -> List[ConversationExample]:
    if name not in DATASET_REGISTRY:
        raise ValueError(f"Unknown dataset '{name}'. Choose from: {list(DATASET_REGISTRY)}")
    return DATASET_REGISTRY[name](
        split=split,
        max_examples=max_examples,
        num_distractors=num_distractors,
        seed=seed,
    )

print('Real dataset loaders ready. Available:', list(DATASET_REGISTRY.keys()))

---
## 8. Full Pipeline

Configure the run below, then execute the cells in order.

| Parameter | Options |
|-----------|--------|
| `DATASET` | `'synthetic'`, `'soda'`, `'blended_skill_talk'`, `'hh_rlhf'` |
| `MODEL` | `'all-MiniLM-L6-v2'` (fast, 384-dim), `'all-mpnet-base-v2'` (stronger) |
| `NUM_DISTRACTORS` | e.g. `9` → 1-in-10 pool; `19` → 1-in-20 pool |

In [ ]:
# ── Configuration (edit these) ────────────────────────────────────────────────
DATASET       = 'synthetic'          # 'synthetic' | 'soda' | 'blended_skill_talk' | 'hh_rlhf'
MODEL         = 'all-MiniLM-L6-v2'  # sentence-transformers model
NUM_DISTRACTORS = 9                  # distractors per example
MAX_EXAMPLES  = 300                  # used for real datasets only
SPLIT         = 'test'
# ─────────────────────────────────────────────────────────────────────────────
print(f'Dataset: {DATASET} | Model: {MODEL} | Pool size: {NUM_DISTRACTORS + 1}')

In [ ]:
# Step 1 — Load dataset
print(f"\n[1/5] Loading dataset '{DATASET}'  (distractors: {NUM_DISTRACTORS})")

if DATASET == 'synthetic':
    examples = build_dataset(num_distractors=NUM_DISTRACTORS)
    print(f'      Built {len(examples)} synthetic examples across 10 domains')
else:
    info = DATASET_INFO.get(DATASET, '')
    if info:
        print(f'      {info}')
    try:
        examples = load_real_dataset(
            name=DATASET,
            split=SPLIT,
            max_examples=MAX_EXAMPLES,
            num_distractors=NUM_DISTRACTORS,
        )
    except Exception as e:
        print(f'\n  ERROR loading dataset: {e}')
        print('  Falling back to synthetic dataset.')
        examples = build_dataset(num_distractors=NUM_DISTRACTORS)

pool_size = get_pool_size(examples)
print(f'      {len(examples)} examples | pool size: {pool_size} candidates each')

In [ ]:
# Step 2 — Prepare shuffled candidate pools
print('\n[2/5] Preparing candidate pools (shuffled)')
contexts = []
candidates_list = []
gold_indices = []

for ex in examples:
    candidates, gold_idx = ex.candidates(shuffle=True)
    contexts.append(ex.context)
    candidates_list.append(candidates)
    gold_indices.append(gold_idx)

print(f'      Ready: {len(contexts)} contexts, {sum(len(c) for c in candidates_list)} candidate texts total')

In [ ]:
# Step 3 — Embed
print(f"\n[3/5] Embedding with '{MODEL}'")
embedder = Embedder(model_name=MODEL)

print('      Embedding contexts...')
context_embs = embedder.embed_contexts(contexts)
print('      Embedding candidates...')
candidate_embs_list = embedder.embed_candidates_batched(candidates_list)

print(f'      context_embs shape: {context_embs.shape}')
print(f'      candidate_embs_list[0] shape: {candidate_embs_list[0].shape}')

In [ ]:
# Step 4 — Rank
print('\n[4/5] Ranking candidates by cosine similarity')
results = rank_all(context_embs, candidate_embs_list, candidates_list, gold_indices)
print(f'      Ranked {len(results)} examples')

In [ ]:
# Step 5 — Evaluate
print('\n[5/5] Computing metrics')

k_values = [1, 2, 3, 5] if pool_size >= 5 else [1, 2, 3]
metrics = compute_all_metrics(results, k_values=k_values)
random_mrr = baseline_random_rank(pool_size)

print_metrics_table(metrics, label=f'EMBEDDING RANKER  (pool={pool_size})')

random_metrics = {'MRR': random_mrr}
for k in k_values:
    random_metrics[f'R@{k}'] = round(k / pool_size, 4)
print_metrics_table(random_metrics, label=f'RANDOM BASELINE   (pool={pool_size})')

print('\n  LIFT OVER RANDOM:')
for key in metrics:
    lift = (metrics[key] - random_metrics[key]) / random_metrics[key] * 100
    arrow = '^' if lift > 0 else 'v'
    print(f'    {key:<6} [{arrow}] {lift:+.1f}%  ({random_metrics[key]:.4f} -> {metrics[key]:.4f})')

dist = rank_distribution(results)
print(f'\nRANK DISTRIBUTION  (N={len(results)} examples):')
print_rank_distribution(dist, len(results))

In [ ]:
# Qualitative examples
print(f"\n{'='*70}")
print('QUALITATIVE EXAMPLES')
print('='*70)

indices = list(range(len(examples)))
random.shuffle(indices)

for idx in indices[:3]:
    ex = examples[idx]
    res = results[idx]
    print(f'\n--- Example {idx + 1} (gold rank: {res.gold_rank}) ---')
    print(f'CONTEXT:\n  {ex.context[:300]}{"..." if len(ex.context) > 300 else ""}')
    print(f'\nGOLD RESPONSE (rank {res.gold_rank}, score {res.gold_score:.4f}):')
    print(f'  {ex.gold[:200]}')
    print(f'\nTOP-3 RANKED CANDIDATES:')
    for rank, (cand, score) in enumerate(
        zip(res.ranked_candidates[:3], res.ranked_scores[:3]), 1
    ):
        marker = ' <-- GOLD' if cand == ex.gold else ''
        print(f'  [{rank}] (score {score:.4f}){marker}')
        print(f'       {cand[:160]}...')

In [ ]:
# Hardest cases
worst = sorted(results, key=lambda r: r.gold_rank, reverse=True)[:3]
worst_idx = [results.index(r) for r in worst]

print(f"\n{'='*70}")
print('HARDEST CASES (gold ranked lowest):')
print('='*70)
for idx, res in zip(worst_idx, worst):
    ex = examples[idx]
    print(f'\n  Context: {ex.context[:160]}...')
    print(f'  Gold response (rank {res.gold_rank}, score {res.gold_score:.4f}):')
    print(f'    {ex.gold[:180]}...')
    print(f'  Top-1 response (score {res.ranked_scores[0]:.4f}):')
    print(f'    {res.ranked_candidates[0][:180]}...')

---
## 9. Experiments

Try different configurations and compare results.

In [ ]:
# Compare two models on synthetic data
configs = [
    {'model': 'all-MiniLM-L6-v2',  'distractors': 9},
    {'model': 'all-MiniLM-L6-v2',  'distractors': 19},
    # {'model': 'all-mpnet-base-v2', 'distractors': 9},  # slower but stronger
]

all_results = {}
for cfg in configs:
    label = f"{cfg['model']}  pool={cfg['distractors']+1}"
    print(f'\n--- Running: {label} ---')
    exs = build_dataset(num_distractors=cfg['distractors'])
    ctxs, cands, golds = [], [], []
    for ex in exs:
        pool, gi = ex.candidates(shuffle=True)
        ctxs.append(ex.context)
        cands.append(pool)
        golds.append(gi)
    emb = Embedder(model_name=cfg['model'])
    ctx_e = emb.embed_contexts(ctxs)
    cand_e = emb.embed_candidates_batched(cands)
    res = rank_all(ctx_e, cand_e, cands, golds)
    ps = get_pool_size(exs)
    kv = [1, 2, 3, 5] if ps >= 5 else [1, 2, 3]
    m = compute_all_metrics(res, k_values=kv)
    all_results[label] = m
    print_metrics_table(m, label=label)

print('\n\nSummary:')
for label, m in all_results.items():
    mrr = m['MRR']
    r1  = m.get('R@1', '-')
    print(f'  {label:<45}  MRR={mrr:.4f}  R@1={r1:.4f}')

---
## Key Takeaways

1. **MRR and R@k** are the standard metrics for dialogue retrieval; R@1 = exact top-1 accuracy.
2. **Pool size matters**: larger pools (more distractors) make the task harder and reveal model limitations.
3. **all-MiniLM-L6-v2** is fast and accurate for a 10-candidate pool; switch to **all-mpnet-base-v2** for harder settings.
4. The evaluation protocol (gold + distractors from other conversations) matches **Ubuntu Corpus v2 / DSTC7** benchmarks.